In [19]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.utils.validation import check_X_y, check_array


class AdaptiveRandomForest:
    def __init__(self, n_estimators=10, max_depth=None, max_features='sqrt', criterion='gini', tree_dropout=0.2, random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.criterion = criterion
        self.tree_dropout = tree_dropout
        self.random_state = random_state
        self.trees = []
        self.feature_weights = None

    def _calculate_feature_weights(self, X, y):
        from sklearn.feature_selection import mutual_info_classif
        return mutual_info_classif(X, y, random_state=self.random_state)

    def fit(self, X, y):
        X, y = check_X_y(X, y)  # Input validation

        # Initialize feature weights
        self.feature_weights = self._calculate_feature_weights(X, y)
        self.feature_weights /= self.feature_weights.sum()  # Normalize weights

        # Random state for reproducibility
        rng = np.random.default_rng(self.random_state)

        # Residual initialization (set to a vector of zeros)
        residual = np.zeros(y.shape)  # Initialize residuals as zero initially

        # Build trees iteratively
        for _ in range(self.n_estimators):
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                criterion=self.criterion,
                max_features=self.max_features,
                random_state=rng.integers(0, 1000)
            )

            # Select features based on feature weights
            feature_subset = rng.choice(
                X.shape[1],
                size=int(np.sqrt(X.shape[1])),
                replace=False,
                p=self.feature_weights
            )
            tree.fit(X[:, feature_subset], y)  # Train on selected features

            # Append the tree and its feature subset
            self.trees.append((tree, feature_subset))

            # Update residual using predicted probabilities instead of class labels
            tree_preds = tree.predict_proba(X[:, feature_subset])  # Get class probabilities
            residual += tree_preds[:, 1]  # Update residual (adjust as needed for class-specific updating)

    def predict(self, X):
        X = check_array(X)  # Input validation
        predictions = np.zeros((X.shape[0], len(self.trees[0][0].classes_)))  # For softmax aggregation

        # Apply tree dropout
        n_active_trees = int((1 - self.tree_dropout) * len(self.trees))
        rng = np.random.default_rng(self.random_state)

        active_trees = rng.choice(len(self.trees), size=n_active_trees, replace=False)

        for i in active_trees:
            tree, features = self.trees[i]
            preds = tree.predict_proba(X[:, features])  # Get class probabilities
            predictions += preds  # Accumulate the probabilities for each class

        # Softmax aggregation for class prediction
        return np.argmax(predictions, axis=1)

    def score(self, X, y):
        return accuracy_score(y, self.predict(X))


if __name__ == "__main__":
    # Load your dataset (replace 'your_dataset.csv' with the actual dataset path)
    data = pd.read_csv('log2.csv')

    # Extract features and target (assuming the 5th column is the target column)
    X = data.drop(columns=data.columns[4]).values  # Drop the target column
    y = data.iloc[:, 4].values  # Select the 5th column as the target

    # Encode the target column
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)

    # Ensure numeric features (convert categorical features to numeric if needed)
    if not np.issubdtype(X.dtype, np.number):
        X = pd.get_dummies(data.drop(columns=data.columns[4])).values

    # Split the dataset into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.3, random_state=42)
    # Initialize the Adaptive Random Forest model with adjusted parameters
   # Adjusting hyperparameters to try and get accuracy closer to 90%

# Initialize the Adaptive Random Forest model with adjusted parameters
# Initialize the Adaptive Random Forest model with optimized parameters
    arf = AdaptiveRandomForest(
        n_estimators=100,  # Increase the number of trees to 100
        max_depth=10,  # Increase depth to 10 to allow more complex decision boundaries
        tree_dropout=0.1,  # Reduce dropout rate for more trees to contribute
        max_features='sqrt',  # Use sqrt of total features per tree (can adjust for more accuracy)
        random_state=42
    )




    # Train the model
    arf.fit(X_train, y_train)

    # Evaluate the model
    print("Train Accuracy:", arf.score(X_train, y_train))
    print("Test Accuracy:", arf.score(X_test, y_test))

    # Decode predictions back to original labels
    y_pred = arf.predict(X_test)
    y_pred_decoded = le.inverse_transform(y_pred)
    print("Decoded Predictions:", y_pred_decoded)


Train Accuracy: 0.9978200209277991
Test Accuracy: 0.9973550356052899
Decoded Predictions: ['deny' 'allow' 'allow' ... 'drop' 'allow' 'allow']
